# 🚀 Kaggle Worker — Ollama (Qwen 2.5 7B) + Ngrok Static Tunnel

**Quy trình:**
1. Cài Ollama & pull model Qwen 2.5 7B
2. Đợi server sẵn sàng (health check loop)
3. Test inference nhanh (verify GPU hoạt động)
4. Mở Ngrok **static** tunnel — URL không đổi giữa các session
5. Self-test end-to-end qua public URL
6. Heartbeat loop giữ session sống

> ⚠️ **URL Ngrok static domain KHÔNG ĐỔI** → không cần cập nhật `.env` trên Local mỗi lần restart!
> Chỉ cần chạy lại notebook là xong.

In [ ]:
# ─── [1/5] Cài đặt Ollama & Dependencies ────────────────────────────────────
print('🚀 [1/5] Đang cài đặt Ollama & pyngrok...')
!apt-get update -y > /dev/null 2>&1
!apt-get install -y zstd curl > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install --quiet pyngrok httpx
print('✅ Cài đặt thành công!')

In [ ]:
# ─── [2/5] Khởi động Ollama Server & Pull Model ───────────────────────────
import os, time, subprocess, httpx

os.environ['OLLAMA_MODELS']  = '/kaggle/working/ollama_models'
os.environ['OLLAMA_HOST']    = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
# ✅ GPU optimization: force all layers to GPU, limit CPU threads
os.environ['CUDA_VISIBLE_DEVICES']    = '0,1'   # expose both T4 GPUs
os.environ['OLLAMA_NUM_PARALLEL']     = '2'     # allow 2 parallel requests
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'   # focus VRAM on 1 model
os.makedirs('/kaggle/working/ollama_models', exist_ok=True)

print('⚡ [2/5] Khởi động Ollama Server ngầm...')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('⏳ Chờ Ollama API sẵn sàng...', end='')
for i in range(30):
    try:
        res = httpx.get('http://127.0.0.1:11434/api/tags', timeout=3.0)
        if res.status_code == 200:
            print(f' ✅ Ollama sẵn sàng sau {(i+1)*2}s!')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(2)
else:
    raise RuntimeError('❌ Ollama không khởi động được trong 60s. Dừng lại.')

print('📥 [3/5] Đang pull model Qwen 2.5 14B (vui lòng chờ 5-10 phút lần đầu)...')
!ollama pull qwen2.5:14b
print('✅ Model đã sẵn sàng!')

In [ ]:
# ─── [3/5] Kiểm tra Model & Inference ────────────────────────────────────
import httpx, json

r = httpx.get('http://127.0.0.1:11434/api/tags', timeout=10.0)
models = r.json().get('models', [])
print(f"📋 Models đang có: {[m['name'] for m in models]}")

if not any('qwen2.5' in m['name'] for m in models):
    raise RuntimeError('❌ Model qwen2.5:14b CHƯA được pull. Chạy lại Cell [2].')

print('🧪 [3/5] Test inference nhanh...')
test_resp = httpx.post(
    'http://127.0.0.1:11434/api/generate',
    json={'model': 'qwen2.5:14b', 'prompt': 'Reply with exactly one word: Hello', 'stream': False},
    timeout=300.0  # 14B cold-start: lần đầu load ~8.9GB vào VRAM mất 2-4 phút
)
if test_resp.status_code == 200:
    reply = test_resp.json().get('response', '').strip()
    print(f"✅ Inference OK! Response: '{reply}'")
else:
    raise RuntimeError(f"❌ Inference FAILED: {test_resp.status_code} — {test_resp.text[:200]}")

In [ ]:
# ─── [4/5] Kết nối Ngrok Static Tunnel & Self-test End-to-End ─────────────
import time, httpx
from pyngrok import ngrok

# ===== CẤU HÌNH NGROK (static domain — KHÔNG ĐỔI giữa các session) =====
NGROK_AUTHTOKEN = '3Hl420hPJiQrcgDIWpked30M1Ca_7rfpvf1wR4MzLUK4pmMPw'
STATIC_DOMAIN   = 'bondless-immerse-paternal.ngrok-free.dev'
# =========================================================================

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()  # Kill tunnel cũ nếu có
time.sleep(1)

print(f'🔗 [4/5] Đang kết nối ngrok → {STATIC_DOMAIN} ...')
tunnel = ngrok.connect(11434, domain=STATIC_DOMAIN)
PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')

print(f"\n{'='*60}")
print(f"🎉 TUNNEL ĐÃ ĐƯỢC THIẾT LẬP!")
print(f"📌 PUBLIC URL: {PUBLIC_URL}")
print(f"{'='*60}")

# Self-test end-to-end qua PUBLIC URL
print('\n🧪 [5/5] Self-test end-to-end qua public URL...')
BYPASS_HEADERS = {
    'ngrok-skip-browser-warning': 'true',
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/125.0.0.0 Safari/537.36',
    'Accept': 'application/json',
}

# Test 1: health check
try:
    r_health = httpx.get(f'{PUBLIC_URL}/api/tags', headers=BYPASS_HEADERS, timeout=20.0)
    assert r_health.status_code == 200, f'HTTP {r_health.status_code} — {r_health.text[:200]}'
    n_models = len(r_health.json().get('models', []))
    print(f'   ✅ Health check PASSED — {n_models} model(s) loaded')
except Exception as e:
    print(f'   ❌ Health check FAILED: {e}')
    print(f'      → Ngrok chưa forward được. Thử kill & reconnect tunnel.')
    raise

# Test 2: inference qua public URL
try:
    r_inf = httpx.post(
        f'{PUBLIC_URL}/api/generate',
        json={'model': 'qwen2.5:14b', 'prompt': 'Reply with exactly one word: Ready', 'stream': False},
        headers=BYPASS_HEADERS,
        timeout=300.0  # 14B cold-start qua tunnel
    )
    assert r_inf.status_code == 200, f'HTTP {r_inf.status_code} — {r_inf.text[:200]}'
    reply = r_inf.json().get('response', '').strip()
    print(f"   ✅ Inference qua public URL PASSED — Response: '{reply}'")
except Exception as e:
    print(f'   ❌ Inference qua public URL FAILED: {e}')
    raise

print(f"""
{'='*60}
🚀 TẤT CẢ TEST ĐÃ PASS!

✅ Trên máy Local, file .env đã có đúng:
   OLLAMA_HOST="{PUBLIC_URL}"
   OLLAMA_MODEL="qwen2.5:14b"
   USE_VLLM="false"
   OLLAMA_TIMEOUT_SECONDS="300"

💡 Vì dùng NGROK STATIC DOMAIN, URL này KHÔNG THAY ĐỔI.
   Không cần cập nhật .env mỗi lần restart Kaggle!

   Chỉ restart Docker worker nếu cần:
   docker compose restart doc-translation-worker
{'='*60}
""")

# ─── Heartbeat loop — giữ Kaggle session sống ─────────────────────────
count = 0
while True:
    time.sleep(30)
    count += 30
    try:
        r = httpx.get('http://127.0.0.1:11434/api/tags', timeout=5.0)
        n_models = len(r.json().get('models', []))
        print(f'[Heartbeat {count}s] ✅ Ollama OK — {n_models} model(s) | Ngrok: {PUBLIC_URL}')
    except Exception as err:
        print(f'[Heartbeat {count}s] ⚠️  Cảnh báo Ollama: {err}')